# Lab 4 Parte 1 — CNN grietas en hormigón — Vía IA-asistida

**Sesión:** redes convolucionales para inspección visual de patología

**Público:** ingenieros civiles **sin experiencia en programación** · **Entorno:** GitHub Codespaces o `labs/.venv`

## Cómo trabajar (Copilot, Gemini, Cursor, etc.)

1. Abre este repo en **Codespaces** o activa `source labs/.venv/bin/activate`.
2. **Ejecuta** las celdas pre-escritas (carga de datos, gráficos base) en orden.
3. Lee la **guía IA** de la sección: objetivo, qué considerar y el prompt sugerido.
4. Copia el prompt a tu asistente; pega el código generado en la celda **«Aquí coloca tu solución»**.
5. Ejecuta tu celda y la **Autoevaluación**; busca ✅ antes de avanzar.
6. Registra tus prompts en [`prompts_entregados.md`](prompts_entregados.md) (entrega obligatoria).
7. Referencia docente: `cnn_grietas_estructuras_solucion.ipynb` (no distribuir al inicio).

**La IA propone; tú validas** con autoevaluación, gráficos y sentido físico/estructural.

Dataset: [`data/DATOS.md`](data/DATOS.md) — subconjunto Mendeley (2 000 imágenes).


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path('.').resolve()))
from _verificar import (
    verificar_panorama_cnn, verificar_eda_dataset, verificar_augmentation,
    verificar_dataloaders, verificar_arquitectura, verificar_entrenamiento,
    verificar_curvas, verificar_metricas, verificar_casos_locales, resumen_seccion,
)

import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.transforms import (
    Resize, ToTensor, Normalize, RandomHorizontalFlip, RandomRotation, ColorJitter,
)
from PIL import Image

%matplotlib inline
sns.set_theme(style='whitegrid')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Entorno listo | device={device}")


## Contexto del dataset (Mendeley — grietas en hormigón)

| Clase | En obra significa… | Imágenes (subset) |
|-------|-------------------|-------------------|
| **Negative** | Superficie sin grieta visible | 800 train + 200 val |
| **Positive** | Superficie con grieta | 800 train + 200 val |

Imágenes **227×227** RGB; en el lab se redimensionan con `IMAGE_SIZE` para entrenar en CPU.

Detalle: [`data/DATOS.md`](data/DATOS.md).


## Pregunta 1 — Panorama CNN en inspección estructural

Las **CNN** aprenden filtros espaciales (bordes, texturas, grietas) a partir de píxeles.

### 📘 Subpreguntas
- **1.a** ¿Qué ventaja tiene una CNN frente a features manuales para grietas?
- **1.b** ¿Qué papel cumplen convolución y pooling?
- **1.c** ¿Por qué necesitamos miles de imágenes etiquetadas?


#### ✍️ Tu respuesta (opcional, 2–3 líneas)




### 🤖 Guía IA — Sección 1: Panorama CNN

**Objetivo:** Listar los componentes clave de una CNN aplicada a inspección de grietas.

**Tu código debe lograr**
- Definir `COMPONENTES_CNN` como lista con al menos 4 componentes
- Incluir convolución, pooling, activación y capa fully connected
- Imprimir la lista

**Variables obligatorias** (la autoevaluación las busca con estos nombres): `COMPONENTES_CNN`

**Considera en tu prompt** (menciónalo al asistente si hace falta)
- Contexto: clasificación binaria de imágenes de hormigón (con/sin grieta)
- Nombres sugeridos: convolución, pooling, ReLU, flatten, fully connected, dropout

**Prompt sugerido** (copiar al asistente y completar con el contexto de arriba):

```text
Estoy en el Lab 4 (CNN para grietas en hormigón).
Genera código que:
1) defina COMPONENTES_CNN = ["convolución", "pooling", "ReLU", "flatten", "fully connected"]
2) imprima "Componentes CNN del laboratorio:"
3) recorra la lista e imprima cada componente con print(f"  · {c}")
No uses imports nuevos.
```

_Puedes usar GitHub Copilot, Gemini, ChatGPT o Cursor. Si falla la autoevaluación, pide a la IA que corrija usando el mensaje ❌._


In [ ]:
# --- Aquí coloca tu solución (código generado por la IA) ---
# Ejecuta esta celda después de pegar tu código.

# Variables OBLIGATORIAS (la autoevaluación las busca con estos nombres):
#   · COMPONENTES_CNN

# Tu código debe:
#   1. Lista con conv, pooling, activación, fc…
#   2. Imprimir cada componente



In [ ]:
# --- Autoevaluación 1 ---
# Requiere (celda «Aquí coloca tu solución»): `COMPONENTES_CNN`
r = []
try:
    r = verificar_panorama_cnn(COMPONENTES_CNN)
except NameError as err:
    print(f"❌ Ejecuta primero tu celda de solución. Falta: {err}")
    r = [False]
resumen_seccion('1 — Panorama CNN', r)


## Pregunta 2 — EDA del dataset

Antes de entrenar, revisa **balance**, **tamaños** y **ejemplos visuales**.

### 📘 Subpreguntas
- **2.a** ¿Están balanceadas las clases en train y val?
- **2.b** ¿Las imágenes tienen tamaño uniforme (227×227)?
- **2.c** ¿Qué diferencias visuales buscarías entre Positive y Negative?


In [ ]:
# --- PRE-ESCRITO: rutas y conteos ---
RUTA_DATOS = Path("data/cracks_subset")
if not RUTA_DATOS.is_dir():
    raise FileNotFoundError("Ejecuta primero: python _preparar_datos.py o bash labs/setup.sh")

conteos = {}
for split in ("train", "val"):
    conteos[split] = {}
    for cls in ("Negative", "Positive"):
        d = RUTA_DATOS / split / cls
        conteos[split][cls] = len(list(d.glob("*.jpg")))

class_names = ["Negative", "Positive"]
display(pd.DataFrame(conteos))


### 🤖 Guía IA — Sección 2: EDA del dataset

**Objetivo:** Explorar balance de clases, tamaños de imagen y mosaico Positive vs Negative.

**Tu código debe lograr**
- Definir `N_EJEMPLOS_MOSAICO` (2–8) y `N_MUESTRAS_EDA` (20–200)
- Gráfico de barras con conteos train/val por clase (usa `conteos`)
- Histograma de anchos y altos de una muestra de imágenes train
- Mosaico 2×N con ejemplos Negative y Positive

**Variables obligatorias** (la autoevaluación las busca con estos nombres): `N_EJEMPLOS_MOSAICO`, `N_MUESTRAS_EDA`, `conteos`

**Considera en tu prompt** (menciónalo al asistente si hace falta)
- La celda anterior definió `RUTA_DATOS`, `conteos` y `class_names`
- Rutas: RUTA_DATOS / 'train' / 'Negative' y 'Positive'
- Usa matplotlib; no redefinas conteos

**Prompt sugerido** (copiar al asistente y completar con el contexto de arriba):

```text
En Jupyter ya tengo RUTA_DATOS, conteos, class_names y PIL Image.
Genera código que:
1) N_EJEMPLOS_MOSAICO = 4; N_MUESTRAS_EDA = 100
2) bar chart: eje x Negative/Positive, barras train y val con conteos[split][cls]
3) tome N_MUESTRAS_EDA jpg de train (mezcla clases), lea width/height con Image.open
4) histograma anchos y altos (subplots o dos hist)
5) mosaico 2 filas x N_EJEMPLOS_MOSAICO con imshow
6) plt.tight_layout(); plt.show()
```

_Puedes usar GitHub Copilot, Gemini, ChatGPT o Cursor. Si falla la autoevaluación, pide a la IA que corrija usando el mensaje ❌._


In [ ]:
# --- Aquí coloca tu solución (código generado por la IA) ---
# Ejecuta esta celda después de pegar tu código.

# Variables OBLIGATORIAS (la autoevaluación las busca con estos nombres):
#   · N_EJEMPLOS_MOSAICO
#   · N_MUESTRAS_EDA

# Tu código debe:
#   1. Barras de balance + histograma de tamaños
#   2. Mosaico 2 filas (Negative, Positive)



In [ ]:
# --- Autoevaluación 2 ---
# Requiere (celda «Aquí coloca tu solución»): `N_EJEMPLOS_MOSAICO`, `N_MUESTRAS_EDA`, `conteos`
r = []
try:
    r = verificar_eda_dataset(conteos, N_EJEMPLOS_MOSAICO, N_MUESTRAS_EDA)
except NameError as err:
    print(f"❌ Ejecuta primero tu celda de solución. Falta: {err}")
    r = [False]
resumen_seccion('2 — EDA', r)


## Pregunta 3 — Transformaciones y data augmentation

En **train** añadimos variación artificial (flip, rotación, color); en **val** solo resize y normalize.

### 📘 Subpreguntas
- **3.a** ¿Por qué augmentamos solo en train?
- **3.b** ¿Qué riesgo tiene rotar demasiado una grieta?
- **3.c** ¿Por qué normalizamos con media 0.5?


### 🤖 Guía IA — Sección 3: Transformaciones y data augmentation

**Objetivo:** Definir transforms de train (con aumento) y val (sin aleatoriedad) y visualizarlos.

**Tu código debe lograr**
- Definir `IMAGE_SIZE` (64–227) y `AUG_ROTATION` (5–45 grados)
- Crear `train_transform` con RandomHorizontalFlip, RandomRotation, ColorJitter, Resize, ToTensor, Normalize
- Crear `val_transform` solo con Resize, ToTensor, Normalize
- Mostrar imagen original y `N_AUG_MOSTRADOS` versiones aumentadas

**Variables obligatorias** (la autoevaluación las busca con estos nombres): `IMAGE_SIZE`, `AUG_ROTATION`, `train_transform`, `val_transform`, `N_AUG_MOSTRADOS`

**Considera en tu prompt** (menciónalo al asistente si hace falta)
- Normalize([0.5]*3, [0.5]*3) para centrar píxeles
- Para imshow desnormaliza: tensor * 0.5 + 0.5
- Aplica train_transform sobre PIL RGB, no sobre tensor ya normalizado

**Prompt sugerido** (copiar al asistente y completar con el contexto de arriba):

```text
Lab 4 CNN. Tengo RUTA_DATOS, transforms, Resize, ToTensor, Normalize, RandomHorizontalFlip, RandomRotation, ColorJitter.
Genera código que:
1) IMAGE_SIZE = 128; AUG_ROTATION = 15; N_AUG_MOSTRADOS = 4
2) train_transform = Compose([RandomHorizontalFlip(), RandomRotation(AUG_ROTATION), ColorJitter(0.2,0.2), Resize((IMAGE_SIZE,IMAGE_SIZE)), ToTensor(), Normalize([0.5,0.5,0.5],[0.5,0.5,0.5])])
3) val_transform = Compose([Resize((IMAGE_SIZE,IMAGE_SIZE)), ToTensor(), Normalize([0.5,0.5,0.5],[0.5,0.5,0.5])])
4) tome una imagen Positive de train, muestre original + N_AUG_MOSTRADOS aplicando train_transform en loop
5) títulos 'original' y 'aug i'
```

_Puedes usar GitHub Copilot, Gemini, ChatGPT o Cursor. Si falla la autoevaluación, pide a la IA que corrija usando el mensaje ❌._


In [ ]:
# --- Aquí coloca tu solución (código generado por la IA) ---
# Ejecuta esta celda después de pegar tu código.

# Variables OBLIGATORIAS (la autoevaluación las busca con estos nombres):
#   · IMAGE_SIZE
#   · AUG_ROTATION
#   · train_transform
#   · val_transform
#   · N_AUG_MOSTRADOS

# Tu código debe:
#   1. train_transform con flip/rotación/jitter
#   2. val_transform determinista
#   3. Plot original vs aumentadas



In [ ]:
# --- Autoevaluación 3 ---
# Requiere (celda «Aquí coloca tu solución»): `IMAGE_SIZE`, `AUG_ROTATION`, `train_transform`, `val_transform`, `N_AUG_MOSTRADOS`
r = []
try:
    r = verificar_augmentation(IMAGE_SIZE, AUG_ROTATION, train_transform, val_transform, N_AUG_MOSTRADOS)
except NameError as err:
    print(f"❌ Ejecuta primero tu celda de solución. Falta: {err}")
    r = [False]
resumen_seccion('3 — Augmentation', r)


## Pregunta 4 — DataLoaders

### 📘 Subpreguntas
- **4.a** ¿Por qué `shuffle=True` solo en train?
- **4.b** ¿Qué forma tiene un batch de imágenes (N×C×H×W)?


### 🤖 Guía IA — Sección 4: DataLoaders

**Objetivo:** Crear train_loader y val_loader con los transforms de la sección anterior.

**Tu código debe lograr**
- Definir `BATCH_SIZE` (8–64)
- Crear `train_ds`, `val_ds` con ImageFolder y transforms ya definidos
- Crear `train_loader` (shuffle=True) y `val_loader` (shuffle=False)

**Variables obligatorias** (la autoevaluación las busca con estos nombres): `BATCH_SIZE`, `train_loader`, `val_loader`

**Considera en tu prompt** (menciónalo al asistente si hace falta)
- Usa train_transform en train y val_transform en val
- RUTA_DATOS / 'train' y RUTA_DATOS / 'val'
- device ya está definido en el setup

**Prompt sugerido** (copiar al asistente y completar con el contexto de arriba):

```text
En Jupyter tengo RUTA_DATOS, train_transform, val_transform, device.
Genera código que:
1) BATCH_SIZE = 32
2) train_ds = ImageFolder(RUTA_DATOS/"train", transform=train_transform)
3) val_ds = ImageFolder(RUTA_DATOS/"val", transform=val_transform)
4) class_names = train_ds.classes
5) train_loader y val_loader con batch_size=BATCH_SIZE
6) imprima class_names y len(train_ds), len(val_ds)
```

_Puedes usar GitHub Copilot, Gemini, ChatGPT o Cursor. Si falla la autoevaluación, pide a la IA que corrija usando el mensaje ❌._


In [ ]:
# --- Aquí coloca tu solución (código generado por la IA) ---
# Ejecuta esta celda después de pegar tu código.

# Variables OBLIGATORIAS (la autoevaluación las busca con estos nombres):
#   · BATCH_SIZE
#   · train_loader
#   · val_loader

# Tu código debe:
#   1. BATCH_SIZE
#   2. ImageFolder + DataLoader train/val



In [ ]:
# --- Autoevaluación 4 ---
# Requiere (celda «Aquí coloca tu solución»): `BATCH_SIZE`, `train_loader`, `val_loader`
r = []
try:
    r = verificar_dataloaders(BATCH_SIZE, train_loader, val_loader, image_size=IMAGE_SIZE)
except NameError as err:
    print(f"❌ Ejecuta primero tu celda de solución. Falta: {err}")
    r = [False]
resumen_seccion('4 — DataLoaders', r)


## Pregunta 5 — Arquitectura CNN

### 📘 Subpreguntas
- **5.a** ¿Qué hace cada bloque Conv+ReLU+Pool?
- **5.b** ¿Por qué dropout antes de la salida?


### 🤖 Guía IA — Sección 5: Arquitectura CNN

**Objetivo:** Construir una CNN pequeña con dos bloques convolucionales y salida binaria.

**Tu código debe lograr**
- Definir `N_FILTERS` (8–128) y `DROPOUT` (0–0.6)
- Crear `modelo` como nn.Module o Sequential con ≥2 Conv2d y salida 2 clases
- Mover modelo a `device`

**Variables obligatorias** (la autoevaluación las busca con estos nombres): `modelo`, `N_FILTERS`, `DROPOUT`

**Considera en tu prompt** (menciónalo al asistente si hace falta)
- Entrada: 3 canales RGB; salida: 2 clases
- Tras conv+pool la imagen se reduce; usa AdaptiveAvgPool2d((4,4)) o calcula flatten
- Ejemplo: Conv→ReLU→MaxPool dos veces, luego Linear

**Prompt sugerido** (copiar al asistente y completar con el contexto de arriba):

```text
Lab 4 CNN grietas. Ya tengo IMAGE_SIZE, device, class_names.
Genera código PyTorch que:
1) N_FILTERS = 32; DROPOUT = 0.3
2) defina class CrackCNN(nn.Module) con:
   - Conv2d(3,N_FILTERS,3,padding=1), ReLU, MaxPool2d(2)
   - Conv2d(N_FILTERS,N_FILTERS*2,3,padding=1), ReLU, MaxPool2d(2)
   - AdaptiveAvgPool2d((4,4)), Flatten, Linear(N_FILTERS*2*16,64), ReLU, Dropout(DROPOUT), Linear(64,2)
3) modelo = CrackCNN().to(device)
4) imprima modelo
```

_Puedes usar GitHub Copilot, Gemini, ChatGPT o Cursor. Si falla la autoevaluación, pide a la IA que corrija usando el mensaje ❌._


In [ ]:
# --- Aquí coloca tu solución (código generado por la IA) ---
# Ejecuta esta celda después de pegar tu código.

# Variables OBLIGATORIAS (la autoevaluación las busca con estos nombres):
#   · modelo
#   · N_FILTERS
#   · DROPOUT

# Tu código debe:
#   1. N_FILTERS y DROPOUT
#   2. CNN con ≥2 Conv2d y salida 2
#   3. modelo.to(device)



In [ ]:
# --- Autoevaluación 5 ---
# Requiere (celda «Aquí coloca tu solución»): `modelo`, `N_FILTERS`, `DROPOUT`
r = []
try:
    r = verificar_arquitectura(modelo, N_FILTERS, DROPOUT)
except NameError as err:
    print(f"❌ Ejecuta primero tu celda de solución. Falta: {err}")
    r = [False]
resumen_seccion('5 — Arquitectura', r)


## Pregunta 6 — Entrenamiento

### 📘 Subpreguntas
- **6.a** ¿Qué optimizador y loss usas para clasificación multiclase?
- **6.b** ¿Cómo detectas overfitting con train vs val?


In [ ]:
# --- PRE-ESCRITO: funciones de entrenamiento y evaluación ---
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)
    return running_loss / total, correct / total

def eval_epoch(model, loader, criterion, device):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item() * images.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)
    return running_loss / total, correct / total

print("✅ Funciones train_one_epoch / eval_epoch listas.")


### 🤖 Guía IA — Sección 6: Entrenamiento

**Objetivo:** Entrenar la CNN varias épocas y guardar métricas en `history`.

**Tu código debe lograr**
- Definir `N_EPOCHS` (1–20) y `LEARNING_RATE` (1e-4–0.1)
- Usar `criterion = nn.CrossEntropyLoss()` y `optimizer = Adam(modelo.parameters(), lr=LEARNING_RATE)`
- Bucle de épocas llamando `train_one_epoch` y `eval_epoch` (ya definidas arriba)
- Guardar listas en `history` con keys train_loss, val_loss, train_acc, val_acc

**Variables obligatorias** (la autoevaluación las busca con estos nombres): `history`, `N_EPOCHS`, `LEARNING_RATE`

**Considera en tu prompt** (menciónalo al asistente si hace falta)
- Las funciones train_one_epoch y eval_epoch ya están en la celda pre-escrita
- No reentrenes desde cero en la autoevaluación

**Prompt sugerido** (copiar al asistente y completar con el contexto de arriba):

```text
En Jupyter tengo modelo, train_loader, val_loader, device, train_one_epoch, eval_epoch.
Genera código que:
1) N_EPOCHS = 5; LEARNING_RATE = 1e-3
2) criterion = nn.CrossEntropyLoss(); optimizer = torch.optim.Adam(modelo.parameters(), lr=LEARNING_RATE)
3) history = {k: [] for k in ["train_loss","val_loss","train_acc","val_acc"]}
4) for epoch in range(N_EPOCHS): entrenar, evaluar, append a history, imprimir época y métricas
5) al final imprima "✅ Entrenamiento completado.
```

_Puedes usar GitHub Copilot, Gemini, ChatGPT o Cursor. Si falla la autoevaluación, pide a la IA que corrija usando el mensaje ❌._


In [ ]:
# --- Aquí coloca tu solución (código generado por la IA) ---
# Ejecuta esta celda después de pegar tu código.

# Variables OBLIGATORIAS (la autoevaluación las busca con estos nombres):
#   · history
#   · N_EPOCHS
#   · LEARNING_RATE

# Tu código debe:
#   1. N_EPOCHS y LEARNING_RATE
#   2. Bucle de entrenamiento + history



In [ ]:
# --- Autoevaluación 6 ---
# Requiere (celda «Aquí coloca tu solución»): `history`, `N_EPOCHS`, `LEARNING_RATE`
r = []
try:
    r = verificar_entrenamiento(history, N_EPOCHS, LEARNING_RATE)
except NameError as err:
    print(f"❌ Ejecuta primero tu celda de solución. Falta: {err}")
    r = [False]
resumen_seccion('6 — Entrenamiento', r)


## Pregunta 7 — Curvas de entrenamiento

### 📘 Subpreguntas
- **7.a** ¿Val loss sube mientras train baja?
- **7.b** ¿Cuántas épocas serían suficientes en producción?


### 🤖 Guía IA — Sección 7: Curvas de entrenamiento

**Objetivo:** Graficar pérdida y accuracy de train vs validación.

**Tu código debe lograr**
- Gráfico con 2 subplots: loss y accuracy
- Eje x = épocas (1..N_EPOCHS)
- Leyenda train vs val

**Variables obligatorias** (la autoevaluación las busca con estos nombres): `history`, `N_EPOCHS`

**Considera en tu prompt** (menciónalo al asistente si hace falta)
- Usa history y N_EPOCHS de la sección anterior
- No reentrenes el modelo aquí

**Prompt sugerido** (copiar al asistente y completar con el contexto de arriba):

```text
Tengo history (train_loss, val_loss, train_acc, val_acc) y N_EPOCHS.
Genera matplotlib con:
- fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
- ax1: plot loss train y val vs range(1, N_EPOCHS+1)
- ax2: plot acc train y val
- leyendas, títulos, plt.tight_layout(), plt.show()
```

_Puedes usar GitHub Copilot, Gemini, ChatGPT o Cursor. Si falla la autoevaluación, pide a la IA que corrija usando el mensaje ❌._


In [ ]:
# --- Aquí coloca tu solución (código generado por la IA) ---
# Ejecuta esta celda después de pegar tu código.

# Variables OBLIGATORIAS (la autoevaluación las busca con estos nombres):
#   · history
#   · N_EPOCHS

# Tu código debe:
#   1. Subplots loss y accuracy
#   2. Leyenda train/val



In [ ]:
# --- Autoevaluación 7 ---
# Requiere (celda «Aquí coloca tu solución»): `history`, `N_EPOCHS`
r = []
try:
    r = verificar_curvas(history, N_EPOCHS)
except NameError as err:
    print(f"❌ Ejecuta primero tu celda de solución. Falta: {err}")
    r = [False]
resumen_seccion('7 — Curvas', r)


## Pregunta 8 — Métricas en validación

### 📘 Subpreguntas
- **8.a** ¿Qué clase se confunde más?
- **8.b** ¿Un falso positivo (grieta) es más grave que un falso negativo?


### 🤖 Guía IA — Sección 8: Métricas en validación

**Objetivo:** Matriz de confusión y classification_report sobre el conjunto val.

**Tu código debe lograr**
- Obtener `y_true` e `y_pred` en val (usa modelo en eval mode)
- Calcular `acc_val` con accuracy_score
- Mostrar `cm` con confusion_matrix y heatmap seaborn
- Imprimir classification_report con target_names=class_names

**Variables obligatorias** (la autoevaluación las busca con estos nombres): `acc_val`, `cm`

**Considera en tu prompt** (menciónalo al asistente si hace falta)
- Puedes reutilizar eval_epoch o iterar val_loader manualmente
- class_names viene de ImageFolder (Negative, Positive)

**Prompt sugerido** (copiar al asistente y completar con el contexto de arriba):

```text
Tengo modelo, val_loader, device, class_names.
Genera código que:
1) ponga modelo.eval(); recolecte y_true, y_pred en val_loader (sin grad)
2) acc_val = accuracy_score(y_true, y_pred)
3) cm = confusion_matrix(y_true, y_pred)
4) heatmap seaborn de cm con annot=True
5) print(classification_report(y_true, y_pred, target_names=class_names))
6) print(f"Accuracy validación: {acc_val:.3f}")
```

_Puedes usar GitHub Copilot, Gemini, ChatGPT o Cursor. Si falla la autoevaluación, pide a la IA que corrija usando el mensaje ❌._


In [ ]:
# --- Aquí coloca tu solución (código generado por la IA) ---
# Ejecuta esta celda después de pegar tu código.

# Variables OBLIGATORIAS (la autoevaluación las busca con estos nombres):
#   · acc_val
#   · cm

# Tu código debe:
#   1. y_true, y_pred en val
#   2. acc_val, cm y gráficos



In [ ]:
# --- Autoevaluación 8 ---
# Requiere (celda «Aquí coloca tu solución»): `acc_val`, `cm`
r = []
try:
    r = verificar_metricas(acc_val, cm)
except NameError as err:
    print(f"❌ Ejecuta primero tu celda de solución. Falta: {err}")
    r = [False]
resumen_seccion('8 — Métricas', r)


## Pregunta 9 — Casos locales

### 📘 Subpreguntas
- **9.a** ¿La CNN mira la grieta o artefactos (sombra, mancha)?
- **9.b** ¿Qué harías para auditar un falso positivo?


### 🤖 Guía IA — Sección 9: Casos locales

**Objetivo:** Mostrar predicciones sobre imágenes concretas de validación.

**Tu código debe lograr**
- Definir `N_CASOS_MOSTRADOS` ≥ 1
- Elegir imágenes de val_ds o val_loader
- Mostrar imagen, etiqueta real y predicción (class_names)

**Variables obligatorias** (la autoevaluación las busca con estos nombres): `N_CASOS_MOSTRADOS`

**Considera en tu prompt** (menciónalo al asistente si hace falta)
- Desnormaliza imagen para imshow: img * 0.5 + 0.5
- Usa modelo.eval() y softmax o argmax

**Prompt sugerido** (copiar al asistente y completar con el contexto de arriba):

```text
Tengo val_ds, class_names, modelo, device.
Genera código que:
1) N_CASOS_MOSTRADOS = 3
2) modelo.eval()
3) para i in range(N_CASOS_MOSTRADOS): tomar val_ds[i], predecir, mostrar imshow con título "real: X | pred: Y"
4) plt.show()
```

_Puedes usar GitHub Copilot, Gemini, ChatGPT o Cursor. Si falla la autoevaluación, pide a la IA que corrija usando el mensaje ❌._


In [ ]:
# --- Aquí coloca tu solución (código generado por la IA) ---
# Ejecuta esta celda después de pegar tu código.

# Variables OBLIGATORIAS (la autoevaluación las busca con estos nombres):
#   · N_CASOS_MOSTRADOS

# Tu código debe:
#   1. N_CASOS_MOSTRADOS ≥ 1
#   2. imshow con etiqueta real vs predicha



In [ ]:
# --- Autoevaluación 9 ---
# Requiere (celda «Aquí coloca tu solución»): `N_CASOS_MOSTRADOS`
r = []
try:
    r = verificar_casos_locales(N_CASOS_MOSTRADOS)
except NameError as err:
    print(f"❌ Ejecuta primero tu celda de solución. Falta: {err}")
    r = [False]
resumen_seccion('9 — Casos locales', r)


## Pregunta 10 — Reflexión ingeniería

### 📘 Subpreguntas
- **10.a** ¿Dónde desplegarías este modelo en un puente o edificio?
- **10.b** ¿Qué datos adicionales recolectarías en obra?
- **10.c** ¿La CNN sustituye la inspección normativa?


#### ✍️ Tu respuesta (opcional, 3 bullets)

- Cuándo usar CNN en inspección: 
- Riesgo principal (falsos positivos/negativos): 
- Validación humana antes de alertar: 


### 🤖 Guía IA — Sección 10: Reflexión ingeniería

**Objetivo:** Responder brevemente cuándo usar CNN en obra y sus límites.

**Tu código debe lograr**
- Escribir 2–3 líneas en markdown o comentarios
- Mencionar inspección visual, falsos positivos y validación humana

**Considera en tu prompt** (menciónalo al asistente si hace falta)
- No hace falta código Python; respuesta textual opcional
- La autoevaluación de esta sección es solo lectura

**Prompt sugerido** (copiar al asistente y completar con el contexto de arriba):

```text
No generes código. Dame 3 bullets breves en español para un ingeniero civil:
- Cuándo una CNN de grietas aporta valor en obra
- Un riesgo (falsos positivos/negativos)
- Qué validación humana harías antes de confiar en la alerta
```


In [ ]:
# --- Aquí coloca tu solución (código generado por la IA) ---
# Ejecuta esta celda después de pegar tu código.

# Tu código debe:
#   1. Respuesta breve opcional (markdown en celda anterior)

# Nota: La sección 10 no tiene autoevaluación con código.



## Cierre y entrega (vía IA)

### ✍️ Reflexión final (3 frases)
- La variable que más impactó fue ___ porque ___.
- Para reducir costos en planta usaría el modelo para ___.
- Antes de obra real validaría con ___.

### 📋 Bitácora de prompts (obligatorio)

Completa [`prompts_entregados.md`](prompts_entregados.md): por cada sección, copia el prompt enviado, un resumen de la respuesta de la IA y qué aceptaste o rechazaste.

---
**Checklist:** 9 autoevaluaciones en ✅ · EDA, augmentation, curvas y confusión · bitácora en prompts_entregados.md · bitácora entregada · gráficos revisados.
